# WideResNet-50-2: CE versus focal loss

Both experiments call the project's tested training scripts. The model is an ImageNet-pretrained WideResNet-50-2, with early stages frozen, layer3/layer4 trainable, and one 2048-to-3 classifier. There is no 64-D projection in these baselines.

Thesis Table 4.7 (PDF page 101 / printed page 79): CE accuracy **54.3 ± 2.4%**, Parkinson precision **0.67 ± 0.05**, F1 **0.57 ± 0.06**; focal accuracy **62.5 ± 2.1%**, precision **0.80 ± 0.06**, F1 **0.62 ± 0.05**. These are published results, not results generated by this notebook.

The thesis describes conventional noise/rotation/flipping and replacement sampling but omits exact strengths and training settings. Values below are documented project choices. PNG-to-time-colour equivalence and the final weighted ensemble remain unverified. This is a thesis-guided baseline, not a certified exact reproduction.

This notebook is ready for the official **Google Colab kernel in VS Code**. The shared dataset is already on Drive in the Google account **aryakargar85@gmail.com**. The training-code snapshot is also uploaded at `My Drive/ParkinsonResearch/parkinson_training_code_v1.zip`. Use this notebook alone for training: it includes the shared-data setup, so running `00_shared_dataset_colab.ipynb` first is optional. Open this notebook, choose **Select Kernel → Colab → New Colab Server**, sign in to that account, select an available **GPU**, and run cells from top to bottom.

The setup cell mounts Drive, verifies and extracts the shared dataset and model code, installs dependencies, and checks the GPU and input paths. It sets `DATA_ROOT`, `DATASET_DIR` and `MANIFEST` automatically; the training commands already use them. Checkpoints and reports save to `My Drive/ParkinsonResearch/runs/baselines_shared_dataset_v1`. Start with the default one-fold pilot. The same notebook also supports local Jupyter from this repository.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True


def extract_checked(source, destination, expected_sha256):
    if not source.is_file():
        raise FileNotFoundError(f'Missing Drive file: {source}. Check the Google account and whether the code archive upload is complete.')
    local_zip = Path('/content') / source.name
    shutil.copy2(source, local_zip)
    if hashlib.sha256(local_zip.read_bytes()).hexdigest() != expected_sha256:
        raise ValueError(f'Archive checksum differs from this notebook: {source.name}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe archive path: {name}')
        archive.extractall(destination)


DRIVE_DIR = None
if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/ParkinsonResearch')
    DATA_ROOT = Path('/content/ParkinsonData')
    extract_checked(
        DRIVE_DIR / 'datasets/parkinson_dataset_v1.zip', DATA_ROOT,
        '85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6',
    )
    metadata = json.loads((DATA_ROOT / 'dataset_package.json').read_text())
    for name, info in metadata['files'].items():
        target = (DATA_ROOT / name).resolve()
        if not target.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f'Unsafe dataset path: {name}')
        if hashlib.sha256(target.read_bytes()).hexdigest() != info['sha256']:
            raise ValueError(f'Dataset verification failed: {name}')
    PROJECT_DIR = Path('/content/ParkinsonAiResearch')
    extract_checked(DRIVE_DIR / 'parkinson_training_code_v1.zip', PROJECT_DIR,
                    '2dc6906a9fcfa2b59ac6874a15bf7a608fc998b8158f2e4fa8a958e264f912e6')
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_DIR = next((p for p in candidates if (p / 'train.py').is_file()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Open this notebook from the ParkinsonAiResearch repository.')
    DATA_ROOT = PROJECT_DIR

DATASET_DIR = DATA_ROOT / 'datasets'
MANIFEST = DATASET_DIR / 'annotations/fold_manifest.csv'
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Dataset manifest not found: {MANIFEST}')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError('Select Kernel → Colab → New Colab Server, choose a GPU, then rerun setup.')
print('Code directory:', PROJECT_DIR)
print('Dataset directory:', DATASET_DIR)
print('Fold manifest:', MANIFEST)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'local automatic device')
print('Ready. Run the configuration cell, then the two training cells.')


Start with fold 1 to check that both experiments finish. Its few validation images cannot establish which loss is better. Then set `FOLD_END = 25` and rerun the configuration and training cells; `--resume` reuses completed folds. Full comparison requires all 25 fold reports, including the explicit empty-validation report for fold 22.

On Colab, `OUTPUT_BASE` defaults to persistent Drive storage. After a runtime reset, reconnect to a GPU and rerun setup, configuration and the training cells with the same archive and settings. `--resume` continues from the last saved epoch; an interrupted epoch is repeated. Use a new output folder when changing the training recipe. Full runs can use many GB of Drive storage for model and optimizer checkpoints.


In [ ]:
FOLD_START = 1
FOLD_END = 25  # set to 25 after the pilot succeeds
EPOCHS = 15
BATCH_SIZE = 8
SEED = 42
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA_MODE = 'uniform'
AUGMENTATION = 'standard'
SAMPLING = 'class_balanced'
ROTATION_DEGREES = 10
HFLIP_PROB = 0.5
NOISE_STD = 0.01
DEVICE = 'cuda' if torch.cuda.is_available() else 'auto'
OUTPUT_BASE = (DRIVE_DIR / 'runs' / 'baselines_shared_dataset_v1' if IN_COLAB
               else PROJECT_DIR / 'outputs' / 'baselines')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print('Checkpoints and reports:', OUTPUT_BASE)

COMMON = [
    '--fold-start', str(FOLD_START), '--fold-end', str(FOLD_END),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--image-size', '224',
    '--pretrained', '--freeze-policy', 'early', '--label-source', 'consensus',
    '--lr', str(HEAD_LR), '--backbone-lr', str(BACKBONE_LR),
    '--weight-decay', str(WEIGHT_DECAY), '--seed', str(SEED),
    '--augmentation', AUGMENTATION, '--sampling', SAMPLING,
    '--rotation-degrees', str(ROTATION_DEGREES), '--hflip-prob', str(HFLIP_PROB),
    '--noise-std', str(NOISE_STD), '--num-workers', '0', '--device', DEVICE,
    '--manifest', str(MANIFEST), '--root', str(DATA_ROOT), '--resume',
]

def run_loss(loss):
    command = [sys.executable, '-u', 'train_all_folds.py', *COMMON,
               '--loss', loss, '--checkpoint-dir', str(OUTPUT_BASE / loss / 'checkpoints')]
    if loss == 'focal':
        command += ['--focal-gamma', str(FOCAL_GAMMA), '--focal-alpha-mode', FOCAL_ALPHA_MODE]
    subprocess.run(command, check=True)


Run WideResNet + cross-entropy. The first pretrained run downloads the ImageNet weights if needed.

In [ ]:
run_loss('cross_entropy')


Run WideResNet + focal loss with the same architecture, fold schedule, seed, optimizer and budget.

In [ ]:
run_loss('focal')


Pool development predictions. The report contains accuracy, Parkinson precision/recall/F1, macro F1, per-class support and the confusion matrix. A pilot report is marked incomplete. Pooled development results are not the thesis's final weighted test-ensemble result.

In [ ]:
import json
for loss in ('cross_entropy', 'focal'):
    output = OUTPUT_BASE / loss
    command = [sys.executable, 'aggregate_validation.py', '--loss', loss,
               '--predictions-dir', str(output / 'validation_predictions'),
               '--output-dir', str(output / 'reports'), '--manifest', str(MANIFEST),
               '--seed', str(SEED), '--freeze-policy', 'early']
    if loss == 'focal':
        command += ['--focal-alpha-mode', FOCAL_ALPHA_MODE]
    if FOLD_START == 1 and FOLD_END == 25:
        command += ['--require-complete']
    subprocess.run(command, check=True)
    report = json.loads((output / 'reports' / 'oof_validation_summary.json').read_text())
    metrics = report['pooled_oof_metrics']
    print(loss, report['status'], 'OOF images:', report['predicted_oof_images'])
    if metrics:
        print('Accuracy:', metrics['accuracy'], 'Parkinson:', metrics['per_class']['parkinson'],
              'Macro F1:', metrics['macro_f1_all_classes'])


Save a small results bundle to share for the next hyperparameter review. It excludes large model checkpoints, which remain in each loss's `checkpoints` folder. On Colab, both the ZIP and the checkpoints are already saved to Drive; download the ZIP from Drive when needed.

In [ ]:
results_zip = OUTPUT_BASE / 'baseline_results.zip'
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for loss in ('cross_entropy', 'focal'):
        for file in (OUTPUT_BASE / loss).rglob('*'):
            if file.is_file() and file.suffix in {'.json', '.csv'}:
                archive.write(file, file.relative_to(OUTPUT_BASE))
print('Results bundle:', results_zip)
